# Confidence Intervals and Hypothesis Tests on Sales Data (ABA Unit 2, Practical 7)

**Aim:** calculate confidence intervals and perform hypothesis tests (one-sample and two-sample t-tests) on a business dataset.

Run the cells in order (Runtime → Run all). Nothing to install. The note `hypothesis-testing-practical` on Sutra explains every step.

In [1]:
# Step 1. Load the data and take a random sample of 36 days from the Pune Express store (S05)
import numpy as np
import pandas as pd
from scipy import stats

url = ("https://raw.githubusercontent.com/Prathameshv07/ai-ds-notes/main/"
       "content/assets/data/sahyadri-mart-sales.csv")
df = pd.read_csv(url, parse_dates=["date"])
s05 = df[df["store_id"] == "S05"]
rng = np.random.default_rng(2026)
sample = s05["footfall"].to_numpy()[rng.choice(len(s05), size=36, replace=False)]
n, xbar, s = len(sample), sample.mean(), sample.std(ddof=1)
print(f"n = {n}, sample mean {xbar:.1f}, sample SD {s:.1f}")

n = 36, sample mean 1276.6, sample SD 280.3


In [2]:
# Step 2. 95% confidence interval for the mean daily footfall, by formula and with scipy
se = s / np.sqrt(n)
t_star = stats.t.ppf(0.975, df=n - 1)
print(f"t* = {t_star:.3f}, SE = {se:.2f}, margin of error = {t_star * se:.1f}")
print(f"By formula: ({xbar - t_star * se:.1f}, {xbar + t_star * se:.1f})")
lo, hi = stats.t.interval(0.95, df=n - 1, loc=xbar, scale=se)
print(f"With scipy: ({lo:.1f}, {hi:.1f})")
print(f"True mean of all 731 days: {s05['footfall'].mean():.1f}")

t* = 2.030, SE = 46.72, margin of error = 94.8
By formula: (1181.7, 1371.4)
With scipy: (1181.7, 1371.4)
True mean of all 731 days: 1257.3


In [3]:
# Step 3. What does 95% mean? Build 1,000 intervals from 1,000 samples and count the hits
true_mean = s05["footfall"].mean()
values = s05["footfall"].to_numpy()
hits = 0
for _ in range(1_000):
    x = rng.choice(values, size=36, replace=False)
    lo, hi = stats.t.interval(0.95, df=35, loc=x.mean(), scale=x.std(ddof=1) / 6)
    hits += lo <= true_mean <= hi
print(f"Intervals containing the true mean: {hits} of 1,000")

Intervals containing the true mean: 952 of 1,000


In [4]:
# Step 4. One-sample t-test. Head office claims S05 averages 1,300 visitors a day.
# H0: mu = 1300, H1: mu != 1300, alpha = 0.05
res = stats.ttest_1samp(sample, popmean=1300)
print(f"t = {res.statistic:.3f}, two-tailed p-value = {res.pvalue:.4f}")
print("Decision:", "reject H0" if res.pvalue <= 0.05 else "fail to reject H0")

t = -0.502, two-tailed p-value = 0.6189
Decision: fail to reject H0


In [5]:
# Step 5. Two-sample t-test: do display days have higher footfall? (S05, ordinary days only)
ordinary = s05[s05["is_festival"] == 0]
with_d = ordinary.loc[ordinary["display"] == 1, "footfall"]
without = ordinary.loc[ordinary["display"] == 0, "footfall"]
res = stats.ttest_ind(with_d, without, equal_var=False)          # Welch's t-test
print(f"Display days: n = {len(with_d)}, mean {with_d.mean():.1f} | other days: n = {len(without)}, mean {without.mean():.1f}")
print(f"Difference {with_d.mean() - without.mean():.1f} visitors, t = {res.statistic:.3f}, p-value = {res.pvalue:.4f}")
print("Decision at alpha 0.05:", "reject H0" if res.pvalue <= 0.05 else "fail to reject H0")

Display days: n = 222, mean 1261.8 | other days: n = 477, mean 1226.4
Difference 35.4 visitors, t = 1.922, p-value = 0.0553
Decision at alpha 0.05: fail to reject H0


In [6]:
# Step 6. Two-sample t-test: discount days vs no-discount days, with a CI for the difference
disc = ordinary.loc[ordinary["discount_pct"] > 0, "footfall"]
none = ordinary.loc[ordinary["discount_pct"] == 0, "footfall"]
res = stats.ttest_ind(disc, none, equal_var=False)
ci = res.confidence_interval(0.95)
print(f"Discount days: n = {len(disc)}, mean {disc.mean():.1f} | no discount: n = {len(none)}, mean {none.mean():.1f}")
print(f"Difference {disc.mean() - none.mean():.1f} visitors, 95% CI ({ci.low:.1f}, {ci.high:.1f})")
print(f"t = {res.statistic:.2f}, p-value = {res.pvalue:.2e}")

Discount days: n = 310, mean 1311.0 | no discount: n = 389, mean 1179.2
Difference 131.8 visitors, 95% CI (99.1, 164.5)
t = 7.92, p-value = 1.18e-14
